In [3]:
import pandas as pd
import numpy as np
import os

# Load datasets by stepping out of the scripts folder
df_users = pd.read_csv('../data/raw/users.csv')
df_tx = pd.read_csv('../data/raw/transactions.csv')

# Calculate user component spending aggregates
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])
latest_date = df_tx['transaction_date'].max()

rfm = df_tx.groupby('user_id').agg({
    'transaction_date': lambda x: (latest_date - x.max()).days, # Recency
    'transaction_id': 'count',                                 # Frequency
    'order_value': 'sum'                                       # Monetary value
}).rename(columns={'transaction_date': 'recency', 'transaction_id': 'frequency', 'order_value': 'monetary'})

# Define customer cohorts based on transaction velocity thresholds
rfm['customer_segment'] = np.where(rfm['monetary'] > rfm['monetary'].median(), 'High-Value VIP', 'Standard Shopper')

# Ensure processed folder exists and export clean analytics asset
os.makedirs('../data/processed', exist_ok=True)
rfm.to_csv('../data/processed/customer_segments.csv')
print("Customer metrics calculation complete. Sample rows:")
rfm.head()


Customer metrics calculation complete. Sample rows:


,recency,frequency,monetary,customer_segment
user_id,,,,
10003,21,1,52.91,Standard Shopper
10004,3,1,295.55,High-Value VIP
10012,4,1,70.64,Standard Shopper
10019,11,1,47.80,Standard Shopper
10024,22,1,69.03,Standard Shopper


In [4]:
from scipy import stats
import pandas as pd

# Load datasets by stepping out of the scripts folder
df_users = pd.read_csv('../data/raw/users.csv')
df_tx = pd.read_csv('../data/raw/transactions.csv')

# A/B Testing Validation Simulation: Paid Ads vs Email Marketing Conversion Ticket Sizes
group_ads = df_tx[df_tx['user_id'].isin(df_users[df_users['acquisition_channel'] == 'Paid Ads']['user_id'])]['order_value']
group_email = df_tx[df_tx['user_id'].isin(df_users[df_users['acquisition_channel'] == 'Email Marketing']['user_id'])]['order_value']

t_stat, p_val = stats.ttest_ind(group_ads, group_email, equal_var=False)
print(f"Hypothesis Testing Matrix:\nT-Statistic: {t_stat:.4f} | P-Value: {p_val:.4f}")


Hypothesis Testing Matrix:
T-Statistic: -1.2735 | P-Value: 0.2045
